<a href="https://colab.research.google.com/github/thehmfpk/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review
**Lane:** `fact_content_daily_performance`, month `2026-03` (same lane as ML-04).

Load `building-baselines` + `flyrank/flyrank-data` per `skills/README.md` if working with an assistant.


## 0. Setup — load March 2026 partition directly

In [1]:
import pandas as pd, numpy as np, os
from datasets import load_dataset
from huggingface_hub import HfApi
from google.colab import userdata
from scipy.stats import spearmanr

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
all_files = api.list_repo_files('FlyRank/internship-warehouse', repo_type='dataset')
march_files = [f for f in all_files if 'fact_content_daily_performance' in f
               and 'sample' not in f and '2026-03' in f]
if not march_files:
    print('[WARNING] No files matched - inspect all_files and fix the filter:')
    for f in [x for x in all_files if 'fact_content_daily_performance' in x and 'sample' not in x][:20]:
        print(' ', f)
    raise ValueError('Fix march_files filter above, then re-run.')

dataset = load_dataset('FlyRank/internship-warehouse', data_files={'train': march_files}, split='train')
needed_cols = ['report_date','client_hash_id','content_hash_id',
               'gsc_data_available','gsc_impressions','gsc_clicks','gsc_avg_position']
cols_present = [c for c in needed_cols if c in dataset.column_names]
dataset = dataset.select_columns(cols_present)
raw = dataset.to_pandas()
raw['report_date'] = pd.to_datetime(raw['report_date'])
print(f'Loaded {len(raw)} rows, {len(raw.columns)} columns for March 2026.')


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 9841378 rows, 7 columns for March 2026.


In [2]:
gsc = raw[raw['gsc_data_available'] == True].copy()
print(f'{len(gsc)} of {len(raw)} rows have gsc_data_available == True.')

monthly = (gsc.groupby(['client_hash_id','content_hash_id'])
              .agg(impressions=('gsc_impressions','sum'),
                   clicks=('gsc_clicks','sum'),
                   avg_position=('gsc_avg_position', lambda s: s[s > 0].mean()),
                   days_with_position=('gsc_avg_position', lambda s: (s > 0).sum()))
              .reset_index())
monthly = monthly[(monthly['impressions'] > 0) & (monthly['avg_position'].notna())].copy()
monthly['ctr'] = monthly['clicks'] / monthly['impressions']
print(f'{len(monthly)} content items with usable position + impressions this month.')
monthly.head()


3611061 of 9841378 rows have gsc_data_available == True.
175304 content items with usable position + impressions this month.


,client_hash_id,content_hash_id,impressions,clicks,avg_position,days_with_position,ctr
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.000000,1,0.000000
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,14.129210,31,0.006042
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,9.225529,6,0.000000
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,8.470926,30,0.000000
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,14.859827,31,0.000000


## 1. My rule and its reason codes

**Rule, in plain words:** content that's already ranking decently (good position) but pulling less click-through than similar-position content typically gets is flagged as a CTR/snippet problem, not a ranking problem. Prioritize by how many extra clicks it could realistically pick up if its CTR caught up to what its own position bucket usually gets.

**Reason codes this rule can output:** just one for now — `CTR_FIX_OPPORTUNITY` (one rule, one reason code, per the card). A second reason code isn't added here since a single-rule baseline with one code is what this week asks for; more reason codes belong to next week's model.

**Before trusting the rule, two signal checks it leans on:**

### Signal 1 — CTR vs position (flag-linked: behind the CTR-fix flag)
Expected: CTR should be higher for better (lower-numbered) positions. Verdict computed from a Spearman correlation, not asserted.

In [3]:
position_bins = [0, 3, 6, 10, 20, 50, np.inf]
position_labels = ['1-3','4-6','7-10','11-20','21-50','51+']
monthly['position_bucket'] = pd.cut(monthly['avg_position'], bins=position_bins, labels=position_labels)

signal1_table = monthly.groupby('position_bucket', observed=True).agg(
    n=('ctr','size'), mean_ctr=('ctr','mean'), median_ctr=('ctr','median')
).reset_index()
print('--- Signal 1: CTR by position bucket ---')
print(signal1_table.to_string(index=False))

corr1, pval1 = spearmanr(monthly['avg_position'], monthly['ctr'])
if pval1 >= 0.05:
    verdict1 = 'FALSE'
elif corr1 < 0:
    verdict1 = 'CONFIRMED'
else:
    verdict1 = 'OPPOSITE'
print(f"\nSpearman correlation(position, ctr) = {corr1:.3f}, p = {pval1:.4f}")
print(f'SIGNAL 1 VERDICT: {verdict1}')


--- Signal 1: CTR by position bucket ---
position_bucket     n  mean_ctr  median_ctr
            1-3 13136  0.011010    0.000962
            4-6 38772  0.006128    0.000679
           7-10 42847  0.004262    0.000000
          11-20 32548  0.003303    0.000000
          21-50 34783  0.002289    0.000000
            51+ 13218  0.000979    0.000000

Spearman correlation(position, ctr) = -0.254, p = 0.0000
SIGNAL 1 VERDICT: CONFIRMED


### Signal 2 — content volume (flag-linked: behind quick-win)
Expected: click opportunity (impressions x CTR shortfall vs bucket norm) should scale up with impression volume — the entire premise of prioritizing by volume. Tested with a correlation, not assumed.

In [6]:
expected_ctr_map = signal1_table.set_index('position_bucket')['mean_ctr'].to_dict()

# FIX: position_bucket is a Categorical dtype (from pd.cut). Mapping it directly can leave
# the result as categorical too, which can't be subtracted with numpy ops (the TypeError above).
# Cast both sides to str for the lookup, then force the mapped result to float.
expected_ctr_map_str = {str(k): v for k, v in expected_ctr_map.items()}
monthly['expected_ctr'] = monthly['position_bucket'].astype(str).map(expected_ctr_map_str).astype(float)

monthly['ctr_shortfall'] = (monthly['expected_ctr'] - monthly['ctr']).clip(lower=0)
monthly['extra_clicks_opportunity'] = monthly['impressions'] * monthly['ctr_shortfall']

volume_bins = monthly['impressions'].quantile([0, .25, .5, .75, 1.0]).values
volume_bins[0] = -1
volume_labels = ['Q1_low','Q2','Q3','Q4_high']
monthly['volume_bucket'] = pd.cut(monthly['impressions'], bins=volume_bins, labels=volume_labels, duplicates='drop')

signal2_table = monthly.groupby('volume_bucket', observed=True).agg(
    n=('extra_clicks_opportunity','size'),
    mean_opportunity=('extra_clicks_opportunity','mean'),
    total_opportunity=('extra_clicks_opportunity','sum')
).reset_index()
print('--- Signal 2: click opportunity by volume bucket ---')
print(signal2_table.to_string(index=False))

corr2, pval2 = spearmanr(monthly['impressions'], monthly['extra_clicks_opportunity'])
if pval2 >= 0.05:
    verdict2 = 'FALSE'
elif corr2 > 0:
    verdict2 = 'CONFIRMED'
else:
    verdict2 = 'OPPOSITE'
print(f"\nSpearman correlation(impressions, opportunity) = {corr2:.3f}, p = {pval2:.4f}")
print(f'SIGNAL 2 VERDICT: {verdict2}')

--- Signal 2: click opportunity by volume bucket ---
volume_bucket     n  mean_opportunity  total_opportunity
       Q1_low 44350          0.025365        1124.923779
           Q2 43321          0.214372        9286.813074
           Q3 43810          1.297818       56857.418433
      Q4_high 43823         17.218099      754548.731563

Spearman correlation(impressions, opportunity) = 0.604, p = 0.0000
SIGNAL 2 VERDICT: CONFIRMED


If Signal 1 doesn't come back CONFIRMED, the CTR-fix premise is shaky for this slice and the rule below should be reconsidered. If Signal 2 is FALSE/OPPOSITE, volume isn't a useful prioritization weight here — a clean negative, worth keeping as-is rather than forcing a CONFIRMED.

## 2. Build the ranked queue (writes the CSV)

Score = `extra_clicks_opportunity`. Filter to good position buckets (already ranking) and a minimum impression floor (noise control).

In [7]:
MIN_IMPRESSIONS = 50
GOOD_POSITION_BUCKETS = ['1-3','4-6','7-10','11-20']

candidates = monthly[
    monthly['position_bucket'].isin(GOOD_POSITION_BUCKETS) &
    (monthly['impressions'] >= MIN_IMPRESSIONS) &
    (monthly['ctr_shortfall'] > 0)
].copy()

candidates['reason_code'] = 'CTR_FIX_OPPORTUNITY'
candidates['action'] = 'REWRITE_TITLE_META'
candidates['score'] = candidates['extra_clicks_opportunity']

ranked_queue = candidates.sort_values('score', ascending=False).reset_index(drop=True)

output_cols = ['client_hash_id','content_hash_id','position_bucket','avg_position',
               'impressions','clicks','ctr','expected_ctr','ctr_shortfall',
               'score','reason_code','action']

os.makedirs('work/outputs', exist_ok=True)
ranked_queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f'Wrote {len(ranked_queue)} ranked rows to work/outputs/baseline_action_score.csv')
ranked_queue[output_cols].head(20)


Wrote 67231 ranked rows to work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,position_bucket,avg_position,impressions,clicks,ctr,expected_ctr,ctr_shortfall,score,reason_code,action
0,client_e547b89c05043229,content_8d7d99f109e19aa2,1-3,2.563756,203497,289,0.001420,0.011010,0.009590,1951.490418,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META
1,client_e547b89c05043229,content_0e03de7680314cd5,1-3,2.675217,221310,720,0.003253,0.011010,0.007757,1716.610537,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META
2,client_e547b89c05043229,content_4ffe18112a5642e3,1-3,2.331060,186983,586,0.003134,0.011010,0.007876,1472.672215,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META
3,client_e547b89c05043229,content_ec2e0346994fb5a5,1-3,2.854514,245276,1480,0.006034,0.011010,0.004976,1220.474836,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META
4,client_e547b89c05043229,content_eadb33b5df496f4a,1-3,2.383011,617124,5668,0.009185,0.011010,0.001825,1126.500207,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META
5,client_e547b89c05043229,content_545bb6cc7081ded3,1-3,2.615390,122905,287,0.002335,0.011010,0.008675,1066.177073,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META
6,client_e547b89c05043229,content_9ef3d7516483e665,1-3,2.481596,89229,92,0.001031,0.011010,0.009979,890.406225,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META
7,client_23a62021009f63c4,content_44f34c0a90047651,7-10,7.346909,212404,24,0.000113,0.004262,0.004149,881.358858,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META
8,client_e547b89c05043229,content_306bc78dff1eb683,1-3,1.488604,80821,35,0.000433,0.011010,0.010577,854.834622,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META
9,client_62f4a7e64f5e0096,content_34a70fea29d15f24,4-6,3.219473,143019,43,0.000301,0.006128,0.005827,833.427043,CTR_FIX_OPPORTUNITY,REWRITE_TITLE_META


## 3. Top-20 review

For each of the top 20: action, reason code, confidence note, what would make it wrong.

In [8]:
top20 = ranked_queue.head(20)
for i, row in top20.iterrows():
    days_covered = monthly.loc[monthly['content_hash_id'] == row['content_hash_id'], 'days_with_position'].iloc[0]
    confidence = 'high confidence' if days_covered >= 15 else ('moderate confidence' if days_covered >= 7 else 'low confidence - thin sample')
    print(f"#{i+1} | action={row['action']} | reason_code={row['reason_code']} | score={row['score']:.1f}")
    print(f"  why: position bucket {row['position_bucket']} (avg_position={row['avg_position']:.1f}), "
          f"CTR {row['ctr']:.4f} vs bucket-expected {row['expected_ctr']:.4f}, "
          f"{row['impressions']:.0f} impressions across {days_covered} days with position data.")
    print(f"  confidence: {confidence}")
    print(f"  would be wrong if: the shortfall is a measurement artifact (e.g. a mid-month tracking gap) "
          f"rather than a real snippet/title problem, or if {days_covered} days is too thin to trust the CTR estimate.")
    print()


#1 | action=REWRITE_TITLE_META | reason_code=CTR_FIX_OPPORTUNITY | score=1951.5
  why: position bucket 1-3 (avg_position=2.6), CTR 0.0014 vs bucket-expected 0.0110, 203497 impressions across 29 days with position data.
  confidence: high confidence
  would be wrong if: the shortfall is a measurement artifact (e.g. a mid-month tracking gap) rather than a real snippet/title problem, or if 29 days is too thin to trust the CTR estimate.

#2 | action=REWRITE_TITLE_META | reason_code=CTR_FIX_OPPORTUNITY | score=1716.6
  why: position bucket 1-3 (avg_position=2.7), CTR 0.0033 vs bucket-expected 0.0110, 221310 impressions across 29 days with position data.
  confidence: high confidence
  would be wrong if: the shortfall is a measurement artifact (e.g. a mid-month tracking gap) rather than a real snippet/title problem, or if 29 days is too thin to trust the CTR estimate.

#3 | action=REWRITE_TITLE_META | reason_code=CTR_FIX_OPPORTUNITY | score=1472.7
  why: position bucket 1-3 (avg_position=2.3

## 4. Weak picks + leakage check

**Weak picks:** which top-20 entries look wrong and why — flagged by thin position-day coverage even though impressions clear the floor.

**Leakage check:** confirm no product flags (e.g. a pre-existing `is_flagged`/`trend_direction`-style column) and no future-window data fed into the score.

In [9]:
# --- Weak picks: thin coverage despite passing the impression floor ---
weak = top20[top20['content_hash_id'].isin(
    monthly.loc[monthly['days_with_position'] < 10, 'content_hash_id']
)]
if weak.empty:
    print('No top-20 picks rely on fewer than 10 days of position data this month - no weak picks by this check.')
else:
    print(f'{len(weak)} of the top 20 rely on fewer than 10 days of position data this month:')
    print(weak[['content_hash_id','score','impressions']].to_string(index=False))

# --- Leakage check ---
print('\n--- Leakage check ---')
cols_used_in_score = ['gsc_impressions','gsc_clicks','gsc_avg_position','gsc_data_available']
flag_like_cols = [c for c in raw.columns if any(k in c.lower() for k in
                  ['flag','trend','label','is_declining','is_flagged'])]
print(f"Columns loaded this notebook: {list(raw.columns)}")
print(f"Any product-flag / trend / label-style columns present: {flag_like_cols if flag_like_cols else 'none'}")
print(f"Score built only from: {cols_used_in_score}")

min_date, max_date = raw['report_date'].min(), raw['report_date'].max()
print(f"Date range actually loaded: {min_date.date()} to {max_date.date()}")
assert max_date < pd.Timestamp('2026-04-01'), 'Data extends past March 2026 - future window leaked in!'
print('Confirmed: no columns beyond raw GSC metrics used, and no dates beyond March 2026 loaded.')


No top-20 picks rely on fewer than 10 days of position data this month - no weak picks by this check.

--- Leakage check ---
Columns loaded this notebook: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
Any product-flag / trend / label-style columns present: none
Score built only from: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_data_available']
Date range actually loaded: 2026-03-01 to 2026-03-31
Confirmed: no columns beyond raw GSC metrics used, and no dates beyond March 2026 loaded.
